# Few-shot example inspection

Spot-check each `(full_doc, summary, label)` tuple the few-shot LLM classifier injects into its prompt. Reads sidecars written by `paper2data.few_shot.build_examples` at `artifacts/_fewshot_cache/*.json`.

**Hypotheses this notebook helps decide:**
- (2) summaries are mislabeled or incoherent — page through the per-example viewer.
- (3) n=1 vs n=2 draw disjoint example sets — see the side-by-side diff cell.
- (4) classify prompt overflows `classify_num_ctx` — see the rendered-prompt + token-count cell.

Pull artifacts from HPC first:
```bash
rsync -av <hpc>:paper2data/artifacts/_fewshot_cache/ ../artifacts/_fewshot_cache/
rsync -av <hpc>:paper2data/artifacts/_summary_cache/ ../artifacts/_summary_cache/  # optional
```

In [14]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Make paper2data importable when running from notebooks/ without pip-installing.
REPO = Path('..').resolve()
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

from paper2data.data import load_twenty_newsgroups
from paper2data.few_shot import FewShotExample, render_examples_block

ARTIFACTS = REPO / 'artifacts'
FEWSHOT_DIR = ARTIFACTS / '_fewshot_cache'
SUMMARY_DIR = ARTIFACTS / '_summary_cache'

# Must match conf/llm_classify.yaml — keep in sync if those values change.
SEED = 42
TEST_SIZE = 0.2
DATA_REMOVE = ('headers', 'footers', 'quotes')

print('repo:', REPO)
print('fewshot sidecars:', FEWSHOT_DIR.exists() and sorted(p.name for p in FEWSHOT_DIR.glob('*.json')))

repo: /home/stef/paper2data
fewshot sidecars: ['qwen2.5_7b-instruct__v1__frac0.1__n10__seed0.json', 'qwen2.5_7b-instruct__v1__frac0.1__n1__seed0.json', 'qwen2.5_7b-instruct__v1__frac0.1__n2__seed0.json', 'qwen2.5_7b-instruct__v1__frac0.1__n4__seed0.json', 'qwen2.5_7b-instruct__v1__frac0.25__n10__seed0.json', 'qwen2.5_7b-instruct__v1__frac0.25__n1__seed0.json', 'qwen2.5_7b-instruct__v1__frac0.25__n2__seed0.json', 'qwen2.5_7b-instruct__v1__frac0.25__n4__seed0.json', 'qwen2.5_7b-instruct__v1__frac0.5__n10__seed0.json', 'qwen2.5_7b-instruct__v1__frac0.5__n1__seed0.json', 'qwen2.5_7b-instruct__v1__frac0.5__n2__seed0.json', 'qwen2.5_7b-instruct__v1__frac0.5__n4__seed0.json', 'qwen2.5_7b-instruct__v1__frac0.75__n10__seed0.json', 'qwen2.5_7b-instruct__v1__frac0.75__n1__seed0.json', 'qwen2.5_7b-instruct__v1__frac0.75__n2__seed0.json', 'qwen2.5_7b-instruct__v1__frac0.75__n4__seed0.json', 'qwen2.5_7b-instruct__v1__frac1.0__n10__seed0.json', 'qwen2.5_7b-instruct__v1__frac1.0__n1__seed0.json', 'qwe

## 1. Reproduce the train split used by `llm_classify.py`

`train_idx` in the sidecar indexes into `X_train` produced by this exact `train_test_split` call (see `src/paper2data/llm_classify.py:115`).

In [15]:
ds = load_twenty_newsgroups(subset='all', remove=DATA_REMOVE)
X = np.asarray(ds.X, dtype=object)
y = ds.y
target_names = ds.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED,
)
print(f'X_train: {len(X_train)} docs, {len(target_names)} classes')

X_train: 15076 docs, 20 classes


## 2. Discover sidecars on disk

Filename convention (see `_sidecar_path` in `few_shot.py:64`):
`{model_tag}__{prompt_version}__frac{fraction}__n{n_per_category}__seed{seed}.json`

In [16]:
import re

_PAT = re.compile(r'^(?P<tag>.+?)__(?P<ver>[^_]+)__frac(?P<frac>[0-9.]+)__n(?P<n>\d+)__seed(?P<seed>\d+)\.json$')

def _scan_sidecars(dir_: Path) -> pd.DataFrame:
    if not dir_.exists():
        return pd.DataFrame()
    rows = []
    for p in sorted(dir_.glob('*.json')):
        m = _PAT.match(p.name)
        if not m:
            continue
        payload = json.loads(p.read_text())
        rows.append({
            'path': str(p.relative_to(REPO)),
            'model_tag': m['tag'],
            'prompt_version': m['ver'],
            'fraction': float(m['frac']),
            'n_per_category': int(m['n']),
            'seed': int(m['seed']),
            'n_examples': len(payload),
        })
    return pd.DataFrame(rows)

sidecars = _scan_sidecars(FEWSHOT_DIR)
if sidecars.empty:
    print(f'No sidecars at {FEWSHOT_DIR}. rsync from HPC first.')
sidecars

,path,model_tag,prompt_version,fraction,n_per_category,seed,n_examples
0,artifacts/_fewshot_cache/qwen2.5_7b-instruct__...,qwen2.5_7b-instruct,v1,0.10,10,0,200
1,artifacts/_fewshot_cache/qwen2.5_7b-instruct__...,qwen2.5_7b-instruct,v1,0.10,1,0,20
2,artifacts/_fewshot_cache/qwen2.5_7b-instruct__...,qwen2.5_7b-instruct,v1,0.10,2,0,40
3,artifacts/_fewshot_cache/qwen2.5_7b-instruct__...,qwen2.5_7b-instruct,v1,0.10,4,0,80
4,artifacts/_fewshot_cache/qwen2.5_7b-instruct__...,qwen2.5_7b-instruct,v1,0.25,10,0,200
5,artifacts/_fewshot_cache/qwen2.5_7b-instruct__...,qwen2.5_7b-instruct,v1,0.25,1,0,20
6,artifacts/_fewshot_cache/qwen2.5_7b-instruct__...,qwen2.5_7b-instruct,v1,0.25,2,0,40
7,artifacts/_fewshot_cache/qwen2.5_7b-instruct__...,qwen2.5_7b-instruct,v1,0.25,4,0,80
8,artifacts/_fewshot_cache/qwen2.5_7b-instruct__...,qwen2.5_7b-instruct,v1,0.50,10,0,200
9,artifacts/_fewshot_cache/qwen2.5_7b-instruct__...,qwen2.5_7b-instruct,v1,0.50,1,0,20


## 3. Pick a sidecar to inspect

Edit the filters below to choose which run to walk through.

In [17]:
PICK_FRACTION = 0.25
PICK_N = 2
PICK_SEED = 0

def _load_sidecar(fraction: float, n: int, seed: int) -> list[FewShotExample]:
    if sidecars.empty:
        raise RuntimeError('no sidecars present — rsync first')
    match = sidecars[
        (sidecars.fraction == fraction)
        & (sidecars.n_per_category == n)
        & (sidecars.seed == seed)
    ]
    if match.empty:
        raise KeyError(f'no sidecar for fraction={fraction} n={n} seed={seed}')
    if len(match) > 1:
        print(f'WARNING: {len(match)} sidecars match; using first ({match.iloc[0]["model_tag"]})')
    path = REPO / match.iloc[0]['path']
    raw = json.loads(path.read_text())
    return [FewShotExample(**r) for r in raw]

examples = _load_sidecar(PICK_FRACTION, PICK_N, PICK_SEED)
print(f'loaded {len(examples)} examples ({PICK_N} per class × {len(target_names)} classes expected)')

loaded 40 examples (2 per class × 20 classes expected)


## 4. Per-example viewer

Change `IDX` and re-run the cell to walk through. Set `SHOW_FULL_DOC=True` to print the entire raw post (otherwise truncated).

In [26]:
IDX = 0
SHOW_FULL_DOC = False
DOC_CHARS = 5000

In [38]:

def show_example(examples: list[FewShotExample], idx: int, *, show_full: bool = False, doc_chars: int = 2000) -> None:
    if not 0 <= idx < len(examples):
        raise IndexError(f'idx {idx} out of range [0, {len(examples)})')
    e = examples[idx]
    doc = X_train[e.train_idx]
    doc_words = len(doc.split())
    sum_words = len(e.summary.split())
    print(f'--- example {idx}/{len(examples) - 1} ---')
    print(f'train_idx={e.train_idx}  label={e.label_idx} ({e.label_name})')
    print(f'doc_words={doc_words}  summary_words={sum_words}  ratio={sum_words / max(1, doc_words):.3f}')
    print()
    print('DOC' + (' (full)' if show_full else f' (first {doc_chars} chars)') + ':')
    print(doc if show_full else doc[:doc_chars] + ('...' if len(doc) > doc_chars else ''))
    print()
    print('SUMMARY:')
    print(e.summary)

show_example(examples, IDX, show_full=SHOW_FULL_DOC, doc_chars=DOC_CHARS)
IDX += 1

--- example 10/39 ---
train_idx=9151  label=5 (comp.windows.x)
doc_words=63  summary_words=19  ratio=0.302

DOC (first 5000 chars):

	Several ways are available -- pageview (which I think came
with our system, so I can't point), ghostview (with ghostscript), and
the new xv 3.00 package all do so.  I recommend the latter two,
especially xv if you want to do any conversion.

	I'm not sure offhand, but I will attempt to mail you the
comp.lang.postscript FAQ, which has a list of utilities.

SUMMARY:
Pageview, Ghostview (with Ghostscript), and xv 3.00 for viewing; xv recommended for conversion. Faq for utilities available upon request.


### Optional: ipywidgets slider

Falls back silently if widgets aren't installed.

In [ ]:
try:
    from ipywidgets import IntSlider, Checkbox, interact
    interact(
        lambda idx, full: show_example(examples, idx, show_full=full, doc_chars=2000),
        idx=IntSlider(min=0, max=len(examples) - 1, step=1, value=0),
        full=Checkbox(value=False, description='show full doc'),
    )
except ImportError:
    print('ipywidgets not installed; use the IDX cell above instead')

## 5. Summary table — all examples in the chosen sidecar at a glance

In [34]:
rows = []
for i, e in enumerate(examples):
    doc = X_train[e.train_idx]
    rows.append({
        'i': i,
        'label_idx': e.label_idx,
        'label_name': e.label_name,
        'train_idx': e.train_idx,
        'doc_words': len(doc.split()),
        'summary_words': len(e.summary.split()),
        'summary_head': e.summary[:120].replace('\n', ' '),
    })
pd.DataFrame(rows)

,i,label_idx,label_name,train_idx,doc_words,summary_words,summary_head
0,0,0,alt.atheism,9784,59,6,Earthquakes attributed to homosexuality on tal...
1,1,0,alt.atheism,13105,137,12,Critics dismiss article as anti-Islamic propag...
2,2,1,comp.graphics,3671,71,15,ARM and MIPS R3000 efficiency compared; Softwa...
3,3,1,comp.graphics,4357,20,8,"Software for converting PCX plots to x,y coord..."
4,4,2,comp.os.ms-windows.misc,183,1137,262,"This appears to be a complex, multipart messag..."
5,5,2,comp.os.ms-windows.misc,1036,106,11,Clean mouse rubber ball with isopropyl alcohol...
6,6,3,comp.sys.ibm.pc.hardware,9738,279,45,Troubleshooting modem transfer issues between ...
7,7,3,comp.sys.ibm.pc.hardware,12156,109,13,Seeking device drivers for old Gateway BNC eth...
8,8,4,comp.sys.mac.hardware,7675,40,9,"Murry, buy floptical for better disk readabili..."
9,9,4,comp.sys.mac.hardware,9499,49,7,1304S monitor flaws reported; user experiences...


## 6. Side-by-side: which docs differ between n=1 and n=2?

Direct test of hypothesis (3): a single shared RNG advances differently per class, so n=2 is NOT a superset of n=1.

In [ ]:
def _indices_by_class(examples: list[FewShotExample]) -> dict[str, list[int]]:
    out: dict[str, list[int]] = {}
    for e in examples:
        out.setdefault(e.label_name, []).append(e.train_idx)
    return out

try:
    ex1 = _load_sidecar(PICK_FRACTION, 1, PICK_SEED)
    ex2 = _load_sidecar(PICK_FRACTION, 2, PICK_SEED)
except KeyError as err:
    print(err)
else:
    by1 = _indices_by_class(ex1)
    by2 = _indices_by_class(ex2)
    diff_rows = []
    for cls in target_names:
        a = set(by1.get(cls, []))
        b = set(by2.get(cls, []))
        diff_rows.append({
            'class': cls,
            'n=1 indices': sorted(a),
            'n=2 indices': sorted(b),
            'shared': sorted(a & b),
            'n=1 ⊂ n=2?': a.issubset(b),
        })
    diff = pd.DataFrame(diff_rows)
    n_subset = int(diff['n=1 ⊂ n=2?'].sum())
    print(f'classes where n=2 is a superset of n=1: {n_subset} / {len(diff)}')
    diff

## 7. Rendered prompt preview + token estimate

Direct test of hypothesis (4): is the rendered classify prompt near or over `classify_num_ctx` (2048 in `conf/llm/qwen25_7b.yaml`)?

Token count uses `len(text) / 4` — coarse, but enough to flag obvious overflow.

In [39]:
examples_block = render_examples_block(examples)
categories_block = '\n'.join(f'- {c}' for c in target_names)

classify_template = (REPO / 'conf' / 'prompts' / 'few_shot_v1.yaml').read_text()
# crude extract: split on the 'classify:' YAML key
classify_body = classify_template.split('classify: |', 1)[-1]

# Use the first test doc as a stand-in for {text}.
sample_text = X_test[0]

rendered = (
    f'Categories:\n{categories_block}\n\n'
    f'Here are labeled examples:\n\n{examples_block}\n'
    f'Now classify the following post in the same way.\n\n'
    f'POST:\n{sample_text}\n\nRespond with JSON matching the schema.'
)

approx_tokens = len(rendered) // 4
print(f'chars={len(rendered)}  approx_tokens={approx_tokens}  classify_num_ctx=2048')
if approx_tokens > 2048:
    print('!! likely over classify_num_ctx — Ollama will truncate from the front')
print()
print('--- first 1500 chars of rendered prompt ---')
print(rendered[:1500])
print('...')
print('--- last 800 chars ---')
print(rendered[-800:])

chars=13588  approx_tokens=3397  classify_num_ctx=2048
!! likely over classify_num_ctx — Ollama will truncate from the front

--- first 1500 chars of rendered prompt ---
Categories:
- alt.atheism
- comp.graphics
- comp.os.ms-windows.misc
- comp.sys.ibm.pc.hardware
- comp.sys.mac.hardware
- comp.windows.x
- misc.forsale
- rec.autos
- rec.motorcycles
- rec.sport.baseball
- rec.sport.hockey
- sci.crypt
- sci.electronics
- sci.med
- sci.space
- soc.religion.christian
- talk.politics.guns
- talk.politics.mideast
- talk.politics.misc
- talk.religion.misc

Here are labeled examples:

POST:
Earthquakes attributed to homosexuality on talk.religion.misc.
RESPONSE: {"category": "alt.atheism"}

POST:
Critics dismiss article as anti-Islamic propaganda; calls for factual discussion with references.
RESPONSE: {"category": "alt.atheism"}

POST:
ARM and MIPS R3000 efficiency compared; Software Motion Pictures at 48 fps on DECstation 5000/200.
RESPONSE: {"category": "comp.graphics"}

POST:
Software for 